In [89]:
import numpy as np
import pandas as pd
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler
from sklearn.cross_decomposition import PLSRegression

# Data

### Time Series Data (Y)

In [40]:
time_series = pd.read_excel("./data/TGF N-prod data.xlsx", sheet_name = 1, header = 0)
igg_concentration = pd.DataFrame(
    {
        campaign: time_series[(time_series["Run"] == campaign) & (~time_series["daily igg concentration [g/L]"].isna())]["daily igg concentration [g/L]"].to_list()
        for campaign in time_series["Run"].unique()
    }
)
igg_concentration = igg_concentration.T.reset_index()
igg_concentration.rename(columns={"index": "campaign"}, inplace=True)
igg_concentration.rename(columns={day: f"igg concentration day {day}" for day in igg_concentration.columns[1:]}, inplace=True)
igg_concentration["campaign"] = igg_concentration["campaign"].str.split("_").str[0]
igg_concentration

,campaign,igg concentration day 0,igg concentration day 1,igg concentration day 2,igg concentration day 3,igg concentration day 4,igg concentration day 5,igg concentration day 6,igg concentration day 7,igg concentration day 8,igg concentration day 9,igg concentration day 10,igg concentration day 11,igg concentration day 12,igg concentration day 13,igg concentration day 14
0,GA001,0.067,0.068,0.069,0.081,0.123,0.221,0.362,0.585,0.919,1.274,1.751,2.324,2.761,3.249,3.397
1,GA002,0.066,0.065,0.068,0.078,0.117,0.210,0.338,0.558,0.869,1.176,1.642,2.179,2.735,3.371,3.543
2,GA003,0.069,0.069,0.070,0.076,0.114,0.339,0.337,0.570,0.913,1.328,1.863,2.467,2.941,3.330,3.435
3,GA004,0.069,0.069,0.069,0.076,0.115,0.198,0.332,0.569,0.872,1.210,1.709,2.331,2.949,3.396,3.561
4,GA005,0.070,0.069,0.069,0.079,0.122,0.213,0.356,0.591,0.885,1.263,1.739,2.474,3.037,3.581,3.651
5,GA006,0.069,0.069,0.069,0.074,0.122,0.209,0.359,0.660,0.888,1.297,1.838,2.474,3.011,3.406,3.571
6,GA007,0.070,0.069,0.069,0.079,0.116,0.212,0.354,0.618,0.960,1.374,1.947,2.536,3.019,3.444,3.552
7,GA008,0.069,0.069,0.069,0.080,0.127,0.213,0.355,0.591,0.917,1.263,1.800,2.406,2.984,3.473,3.581
8,GA009,0.069,0.068,0.070,0.080,0.119,0.218,0.366,0.619,0.941,1.328,1.832,2.467,2.993,3.358,3.444
9,GA010,0.069,0.069,0.069,0.077,0.106,0.186,0.338,0.673,0.943,1.328,1.863,2.421,2.984,3.454,3.512


### Static Features Data (X)

In [71]:
static_features = pd.read_excel("./data/TGF N-prod data.xlsx", sheet_name = 0, header = 1)
df = static_features.loc[2:].reset_index(drop = True)
df.rename(columns={"CAMPAIGN": "campaign"}, inplace=True)
num_columns = [
    'base total quantity_1',
    'bioreactor cooling duration',
    # 'cell concentration at  end_1',
    'cell concentration at start_1',
    'copper sulfate concentration added_1',
    'copper sulfate quantity_1',
    # 'final bioreactor weight_1',
    'inoculum volume',
    'inoculum weight_1',
    'media warming time_1',
    'medium addition_1',
    'medium removal_1',
    'process step duration_1',
    'process step duration_2',
    'start bioreactor weight',
    'temperature switch process time_1',
    '1M Sodium Carbonate Solution',
    'COPPER SULFATE 50MM',
    'Cell Boost 7a feed medium',
    'Cell Boost 7b feed medium',
    'Glucose 500 g/L',
    'OPTICHO GROWTH MEDIUM',
]
df[num_columns] = df[num_columns].apply(pd.to_numeric)
df = df[["campaign"] + num_columns]
df

,campaign,base total quantity_1,bioreactor cooling duration,cell concentration at start_1,copper sulfate concentration added_1,copper sulfate quantity_1,inoculum volume,inoculum weight_1,media warming time_1,medium addition_1,...,process step duration_1,process step duration_2,start bioreactor weight,temperature switch process time_1,1M Sodium Carbonate Solution,COPPER SULFATE 50MM,Cell Boost 7a feed medium,Cell Boost 7b feed medium,Glucose 500 g/L,OPTICHO GROWTH MEDIUM
0,GA001,NaN,NaN,NaN,NaN,NaN,210.256,NaN,51,1470.000,...,333,32,1517.00,NaN,1003086,1003508,1003442,1003443,1003434,1003435
1,GA002,NaN,NaN,NaN,NaN,NaN,200.688,NaN,50,1470.000,...,333,32,1518.00,NaN,1003086,1003623,1003560,1003561,1003434,1003510
2,GA003,13.885,130.0,0.70,199.506826,8.115,215.532,215.532,52,1450.192,...,332,32,1599.44,116.0,1004762,1005428,1005295,1005424,1004750,1005159
3,GA004,14.293,99.0,0.72,199.436797,8.061,253.017,253.017,55,1449.706,...,331,32,1595.14,114.0,1004762,1005429,1005295,1005424,1004750,1005160
4,GA005,15.382,133.0,0.65,199.484387,8.055,221.518,221.518,52,1449.553,...,333,32,1595.37,118.0,1004762,1005536,1005328,1005424,1004750,1005273
5,GA006,12.047,105.0,0.65,199.395390,8.101,221.906,221.906,51,1449.314,...,334,32,1597.90,119.0,1004762,1005537,1005328,1005329,1004750,1005310
6,GA007,13.266,94.0,0.71,199.469984,8.102,242.761,242.761,51,1448.384,...,335,32,1595.36,118.0,1004762,1005635,1005483,1005484,1005161,1005327
7,GA008,14.576,131.0,0.69,199.426707,8.069,217.916,217.916,51,1449.842,...,334,32,1596.26,119.0,1004762,1005736,1005483,1005484,1005161,1005451
8,GA009,11.849,120.0,0.65,199.442332,8.074,219.751,219.751,51,1449.424,...,333,32,1597.25,118.0,1005162,1005928,1005636,1005914,1005161,1005482
9,GA010,16.284,152.0,0.68,199.423013,8.096,217.552,217.552,51,1449.745,...,334,32,1596.81,118.0,1005162,1005981,1005636,1005484,1005326,1005634


### Ignore columns with NaN values in campaigns GA001 and GA002

In [72]:
column_with_non_nan_values = [
    'final bioreactor weight_1',
    'inoculum volume',
    'media warming time_1',
    'medium addition_1',
    'process step duration_1',
    'process step duration_2',
    'start bioreactor weight',
    '1M Sodium Carbonate Solution',
    'COPPER SULFATE 50MM',
    'Cell Boost 7a feed medium',
    'Cell Boost 7b feed medium',
    'Glucose 500 g/L',
    'OPTICHO GROWTH MEDIUM',
]
df = df[["campaign"] + [col for col in column_with_non_nan_values if col in df.columns.to_list()]]
df

,campaign,inoculum volume,media warming time_1,medium addition_1,process step duration_1,process step duration_2,start bioreactor weight,1M Sodium Carbonate Solution,COPPER SULFATE 50MM,Cell Boost 7a feed medium,Cell Boost 7b feed medium,Glucose 500 g/L,OPTICHO GROWTH MEDIUM
0,GA001,210.256,51,1470.000,333,32,1517.00,1003086,1003508,1003442,1003443,1003434,1003435
1,GA002,200.688,50,1470.000,333,32,1518.00,1003086,1003623,1003560,1003561,1003434,1003510
2,GA003,215.532,52,1450.192,332,32,1599.44,1004762,1005428,1005295,1005424,1004750,1005159
3,GA004,253.017,55,1449.706,331,32,1595.14,1004762,1005429,1005295,1005424,1004750,1005160
4,GA005,221.518,52,1449.553,333,32,1595.37,1004762,1005536,1005328,1005424,1004750,1005273
5,GA006,221.906,51,1449.314,334,32,1597.90,1004762,1005537,1005328,1005329,1004750,1005310
6,GA007,242.761,51,1448.384,335,32,1595.36,1004762,1005635,1005483,1005484,1005161,1005327
7,GA008,217.916,51,1449.842,334,32,1596.26,1004762,1005736,1005483,1005484,1005161,1005451
8,GA009,219.751,51,1449.424,333,32,1597.25,1005162,1005928,1005636,1005914,1005161,1005482
9,GA010,217.552,51,1449.745,334,32,1596.81,1005162,1005981,1005636,1005484,1005326,1005634


# Preprocessing

In [73]:
X = df.drop(columns=["campaign"])
Y = igg_concentration.drop(columns=["campaign"])

### Scaling

In [74]:
# Standardize features and targets
scaler_x = StandardScaler()
scaler_y = StandardScaler()

X_scaled = scaler_x.fit_transform(X)
Y_scaled = scaler_y.fit_transform(Y)

### LASSO

To isolate which initial conditions are critical for each specific day, a looped LassoCV (Cross-Validated LASSO) is run. This forces uninformative features to have a coefficient of exactly zero, leaving a clean, sparse list of key drivers per day.

In [86]:
day_names = Y.columns
important_drivers_per_day = {}

for i, day in enumerate(day_names):
    y_day = Y_scaled[:, i]
    
    # 3-Fold Cross-Validation is ideal for N=10 to find the best penalty parameter (alpha)
    lasso = LassoCV(cv=3, random_state=42, max_iter=100000).fit(X_scaled, y_day)
    
    # Extract features that didn't get zeroed out
    coefs = lasso.coef_
    active_drivers = [(X.columns[j], coefs[j]) for j in range(len(coefs)) if coefs[j] != 0]
    
    # Sort drivers by absolute impact
    important_drivers_per_day[day] = sorted(active_drivers, key=lambda x: abs(x[1]), reverse=True)


In [87]:
for day in day_names:
    print(f"Top drivers for {day}:", important_drivers_per_day[day])

Top drivers for igg concentration day 0: [('medium addition_1', np.float64(-0.808844954453293)), ('inoculum volume', np.float64(0.07849071032061686)), ('process step duration_1', np.float64(0.036706775062967614))]
Top drivers for igg concentration day 1: [('start bioreactor weight', np.float64(0.5802494676967781)), ('media warming time_1', np.float64(0.32574002226857146)), ('process step duration_1', np.float64(0.21282550665964295))]
Top drivers for igg concentration day 2: [('Cell Boost 7b feed medium', np.float64(0.5469329905241461)), ('process step duration_1', np.float64(-0.15698938515093866)), ('inoculum volume', np.float64(-0.04927404151600285))]
Top drivers for igg concentration day 3: []
Top drivers for igg concentration day 4: []
Top drivers for igg concentration day 5: [('process step duration_1', np.float64(-0.02285652163273758))]
Top drivers for igg concentration day 6: []
Top drivers for igg concentration day 7: [('process step duration_1', np.float64(0.5018902291262817)),

# Partial Least Squares (PLS) Regression

In [ ]:
# Fit PLS with 2 latent components (keeps it conservative for 10 samples)
pls = PLSRegression(n_components=2)
pls.fit(X_scaled, Y_scaled)

# Calculate VIP (Variable Importance in Projection) scores
def calculate_vip(model):
    t = model.x_scores_
    w = model.x_weights_
    q = model.y_loadings_
    p, h = w.shape
    vips = np.zeros((p,))
    s = np.diag(np.matmul(np.matmul(np.matmul(t.T, t), q.T), q)).reshape(h, -1)
    total_s = np.sum(s)
    for i in range(p):
        w_l = np.zeros((h,))
        for l in range(h):
            w_l[l] = np.linalg.norm(w[i, l])**2
        vips[i] = np.sqrt(p * np.sum(s.flatten() * w_l) / total_s)
    return vips

vip_scores = calculate_vip(pls)
vip_summary = pd.DataFrame({'Condition': X.columns, 'VIP_Score': vip_scores})
print(vip_summary.sort_values(by='VIP_Score', ascending=False))


                       Condition  VIP_Score
3        process step duration_1   1.305340
2              medium addition_1   1.063250
10               Glucose 500 g/L   1.058828
5        start bioreactor weight   1.054370
11         OPTICHO GROWTH MEDIUM   1.051153
8      Cell Boost 7a feed medium   1.048211
1           media warming time_1   1.047172
7            COPPER SULFATE 50MM   1.043919
6   1M Sodium Carbonate Solution   1.038651
9      Cell Boost 7b feed medium   1.021306
0                inoculum volume   0.648775
4        process step duration_2   0.000000
